# Candidate Generator v2.3 — resumable Colab retrieval

Bu notebook yalnız candidate retrieval içindir. Önce küçük bir Colab smoke testi yapılır. Smoke çıktısı birlikte incelenmeden `AUTHORIZE_FULL_RUN` açılmaz. Tam retrieval bile duplicate kararı, family, train/validation/test split veya model eğitimi yapmaz.

Her leaf'in pHash ve descriptor çıktısı, her source-group retrieval parçası ve final dosyalar SHA-256 ile kaydedilir. Colab kesilirse aynı komut tamamlanmış artifact'ları doğrular ve atlar.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

assert 'google.colab' in sys.modules, 'Bu notebook Colab için hazırlanmıştır.'
from google.colab import drive
drive.mount('/content/drive')
print('Drive bağlandı.')

## 1. Proje ve veri yolları

Yalnız `DATASET_ROOT` satırını kendi Drive klasörüne göre değiştireceğiz. Dataset klasörünün doğrudan sekiz ana klasörü (`ALL`, `Brain Cancer`, vb.) içermesi gerekir. Output Drive'da kalır; runtime kapanınca silinmez.

In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
DATASET_ROOT = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer/Multi Cancer')  # BİRLİKTE DÜZELTECEĞİZ
OUTPUT_BASE = Path('/content/drive/MyDrive/NNGA_OUTPUTS')
SMOKE_OUTPUT = OUTPUT_BASE / 'candidate_generator_v2_3_colab_resume_smoke_v1'
FULL_OUTPUT = OUTPUT_BASE / 'candidate_generator_v2_3_full_retrieval_v1'
EXACT_REPORT = OUTPUT_BASE / 'exact_duplicates_v1.json'

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    print('Proje zaten var; otomatik git pull yapılmadı. Mevcut kod contract ile korunacak.')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
print('Git commit:', commit)
print('Dataset:', DATASET_ROOT)
print('Smoke output:', SMOKE_OUTPUT)
print('Full output:', FULL_OUTPUT)
assert (PROJECT_ROOT / 'config' / 'candidate_generator_v2_3.json').is_file()
assert DATASET_ROOT.is_dir(), 'DATASET_ROOT yanlış veya dataset Drive’a yüklenmemiş.'
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

## 2. GPU ve ortam kontrolü

Colab menüsünden **Runtime → Change runtime type → T4 GPU** seçilmiş olmalı. İlk smoke için mevcut Colab paketleri kaydedilir. Tam çalışmadan önce smoke raporundaki sürümleri beraber değerlendireceğiz; ortam değişirse mevcut contract ile resume reddedilir.

In [ ]:
import torch, torchvision, numpy, PIL
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('NumPy:', numpy.__version__, '| Pillow:', PIL.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
assert hasattr(numpy, 'bitwise_count'), 'NumPy >=2.0 gerekli; burada dur ve Codex’e sürüm çıktısını gönder.'
assert torch.cuda.is_available(), 'GPU seçilmemiş; T4 runtime aç.'

## 3. Exact-duplicate manifest

GitHub'a dataset artifact'ları yüklenmediği için Colab aynı SHA-256 exact-duplicate manifestini datasetten bir kez yeniden üretir. Runner, mutlak Drive yolu farklı olsa bile normalize edilmiş grup hash'inin yerelde doğrulanan manifestle aynı olmasını zorunlu tutar.

In [ ]:
if not EXACT_REPORT.exists():
    subprocess.run([
        sys.executable, str(PROJECT_ROOT / 'src' / 'audit_exact_duplicates.py'),
        '--data-dir', str(DATASET_ROOT),
        '--output', str(EXACT_REPORT),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Exact manifest zaten var; runner hash ile doğrulayacak:', EXACT_REPORT)

## 4. Zorunlu Colab smoke testi

Bu hücre yalnız 26 leaf × 2 = 52 görüntü çalıştırır. Aynı hücre tekrar çalıştırıldığında tamamlanmış artifact'lar hash ile doğrulanıp atlanmalıdır. Smoke sonucu Codex ile incelenmeden tam çalışma açılmaz.

In [ ]:
smoke_command = [
    sys.executable, str(PROJECT_ROOT / 'src' / 'run_candidate_generator_v2_3_full_retrieval.py'),
    '--scope', 'smoke',
    '--smoke-images-per-leaf', '2',
    '--data-dir', str(DATASET_ROOT),
    '--exact-duplicates', str(EXACT_REPORT),
    '--output-dir', str(SMOKE_OUTPUT),
    '--batch-size', '16',
    '--workers', '4',
    '--robust-block-size', '256',
    '--phash-block-size', '128',
    '--seed', '42',
    '--stop-after', 'final',
]
print(' '.join(smoke_command))
subprocess.run(smoke_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
smoke_report = json.loads((SMOKE_OUTPUT / 'run_report.json').read_text(encoding='utf-8'))
print('Durum:', smoke_report['status'])
print('Görüntü:', smoke_report['input']['selected_image_count'])
print('Aday:', smoke_report['candidate_counts']['union_pairs'])
print('Çift/görüntü:', round(smoke_report['candidate_counts']['union_pairs_per_image'], 3))
print('Cross-group:', smoke_report['integrity']['cross_source_group_pairs'])
print('Eksik exact:', smoke_report['integrity']['missing_exact_duplicate_pairs'])
print('Codex’e ver:', SMOKE_OUTPUT / 'run_report.json')
print('Codex’e ver:', SMOKE_OUTPUT / 'hash_manifest.json')

## 5. TAM retrieval — şimdilik kilitli

Bu hücreyi smoke raporunu birlikte inceleyip açıkça karar vermeden çalıştırma. Tam çalışma 130.000 görüntünün descriptor/pHash ve source-group içi adaylarını üretir. Bittiğinde bile split veya model eğitimi başlamaz. Colab kesilirse aynı hücreyi yeniden çalıştırmak resume işlemidir.

In [ ]:
AUTHORIZE_FULL_RUN = False  # Smoke raporunu birlikte onayladıktan sonra True olacak
assert AUTHORIZE_FULL_RUN, 'Tam retrieval kilitli; önce smoke raporunu Codex ile incele.'

full_command = [
    sys.executable, str(PROJECT_ROOT / 'src' / 'run_candidate_generator_v2_3_full_retrieval.py'),
    '--scope', 'full',
    '--confirm-full-run', 'FROZEN_V2_3_130000',
    '--data-dir', str(DATASET_ROOT),
    '--exact-duplicates', str(EXACT_REPORT),
    '--output-dir', str(FULL_OUTPUT),
    '--batch-size', '16',
    '--workers', '4',
    '--robust-block-size', '256',
    '--phash-block-size', '128',
    '--seed', '42',
    '--stop-after', 'final',
]
print(' '.join(full_command))
subprocess.run(full_command, cwd=PROJECT_ROOT, check=True)

## 6. Tam çalışma bittikten sonra

Başarılı final durumda `run_report.json` içindeki status `candidate_generator_v2_3_full_retrieval_complete` olur. Codex'e önce `run_report.json` ve `hash_manifest.json` verilir. `candidate_pairs.csv/.npz`, `run_contract.json`, `stage_manifest.json` ve `stages/` klasörü Drive'da silinmeden korunur.

In [ ]:
if (FULL_OUTPUT / 'run_report.json').is_file():
    full_report = json.loads((FULL_OUTPUT / 'run_report.json').read_text(encoding='utf-8'))
    print('Durum:', full_report['status'])
    print('Aday çift:', full_report['candidate_counts']['union_pairs'])
    print('Çift/görüntü:', full_report['candidate_counts']['union_pairs_per_image'])
    print('Cross-group:', full_report['integrity']['cross_source_group_pairs'])
    print('Eksik exact:', full_report['integrity']['missing_exact_duplicate_pairs'])
    print('Codex’e ver:', FULL_OUTPUT / 'run_report.json')
    print('Codex’e ver:', FULL_OUTPUT / 'hash_manifest.json')
else:
    print('Tam çalışma henüz tamamlanmadı veya başlatılmadı.')